# Datamine MANOVA Process Example

This notebook demonstrates how to configure and run the **`manova`** process wrapper in `dmstudio`.

## Process Description

## MANOVA

Multivariate analysis of variance for an unbalanced and balanced design. It is a more sophisticated technique than the one way analysis of variance ([ANOVA1](<anova1.md>)), in that up to 10 groups can be tested.

**MANOVA** is a procedure for comparing multivariate sample means. As a multivariate procedure, it is used when there are two or more dependent variables, and is typically followed by significance tests involving individual dependent variables separately.

**MANOVA** Requires an input file that contains up to 10 groups of replicate observations defined by keyfields. Note, a minimum of two compulsory keyfields are required. Absent data values are ignored.

The groups of samples which have been classified on geological, sampling, or analytical criteria are checked against each other using the Fisher F statistic to test the significance of the variance between individual samples and group means. The ratios between group variances are calculated to have a minimum value of 1 (nul hypothesis) where the samples between the groups are considered not to be significantly different from each other. Fisher F values are compared with published results from tables against the relevant degrees of freedom. Calculated F values greater than those given in the tables demonstrate that there are significant differences between the groupings.

The version of **MANOVA** provided by your product is for a balanced or unbalanced design. For example, in a balanced design, samples are present for all groupings and sub-groupings. In an unbalanced design, samples can be absent from their allocated positions ie. when a sample is split into A and B, which in turn are split again into A1, A2 and B1, B2, this is balanced. However if B2 is lost, thus not present in the analysis of variance, then the sampling is unbalanced.

### File Handling

Samples must be sorted on defined keyfields. Using Fisher F Tables, check the two degrees of freedom between the group pairs (**DF1** and **DF2**) and look for the corresponding F value in the table which corresponds to the cross over point. If this value exceeds the **MANOVA** value for a given confidence level then the nul hypothesis does not hold, ie, the sample groupings are significantly different from each other for the defined level of confidence.

The input file (&**IN**) must be sorted on the keyfields which define the sample classification groups, for example, vein, sample location, laboratory and sample split.

### Input Files:

* **in** (Undefined):
  Input file, sorted on required keyfields.
  Required=Yes

### Output Files:

### Fields:

* **value** (Numeric : IN):
  Field for analysis of variance.
  Default=Undefined
  Required=Yes

* **keys** (Undefined : Undefined):
  Keyfield 1 for replicate observations.
  Default=Undefined
  Required=No

### Parameters:

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('manova')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute manova
print("Running manova...")
dm_cmd.manova(
    in_i='t_assays',  # required
    value_f='optional',  # required
    # keys_f=['BHID'],  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("manova execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_manova_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")